# **Sel 1: Markdown**

# **1. Load Dataset Kaggle & Vocabulary**

# **Sel 2: Code**

In [13]:
import pandas as pd
import numpy as np

# Load semua bahan baku
df_kaggle = pd.read_csv('tech_internship_applications.csv')
df_job_large = pd.read_csv('job_dataset.csv')
df_vocab = pd.read_csv('skill_vocabulary.csv')

print(f"Kaggle Data: {df_kaggle.shape}")
print(f"Large Job Data: {df_job_large.shape}")

Kaggle Data: (300, 10)
Large Job Data: (1068, 7)


# **Sel 3: Markdown**

**2. Handle Missing Values & Filter Khusus Data Intern**

# **Sel 4: Code**

In [14]:
# Kita perluas jaring kata kuncinya
keywords = 'intern|trainee|fresher|entry|junior|student'

# 1. Cek di kolom Title
mask_title = df_job_large['Title'].str.contains(keywords, case=False, na=False)

# 2. Cek di kolom ExperienceLevel (kalau ada)
if 'ExperienceLevel' in df_job_large.columns:
    mask_exp = df_job_large['ExperienceLevel'].str.contains(keywords, case=False, na=False)
    # Gabungkan: Ambil jika ada di Title ATAU ada di ExperienceLevel
    filter_intern = mask_title | mask_exp
else:
    filter_intern = mask_title

# Eksekusi filter
df_job_intern = df_job_large[filter_intern].copy()

print(f"Data Intern Tambahan yang didapat: {len(df_job_intern)} baris")
display(df_job_intern.head(3))

Data Intern Tambahan yang didapat: 434 baris


,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...


# **Sel 5: Markdown**

## **3. Penyelarasan Kolom & Penggabungan**

# **Sel 6: Code**

In [15]:
# 1. Siapkan data Kaggle
df_k = pd.DataFrame()
df_k['job_title'] = df_kaggle['Internship_Domain']
df_k['raw_skills'] = df_kaggle['Skills'].fillna('')

# 2. Siapkan data Large Job (hasil filter intern)
df_j = pd.DataFrame()
df_j['job_title'] = df_job_intern['Title']
df_j['raw_skills'] = df_job_intern['Skills'].fillna('')

# 3. Gabungkan jadi satu Super-Dataset Global
df_global = pd.concat([df_k, df_j], ignore_index=True)
print(f"Total Data Global Gabungan: {len(df_global)} baris")

Total Data Global Gabungan: 734 baris


# **Sel 7: Markdown**

### **4. Feature Engineering (Role & Skills Count)**

# **Sel 8: Code**

In [16]:
def map_27_roles(text):
    text = str(text).lower()
    # Data & AI
    if 'data scientist' in text or 'data science' in text: return 'Data Scientist'
    elif 'data analyst' in text or 'analytics' in text: return 'Data Analyst'
    elif 'data engineer' in text: return 'Data Engineer'
    elif 'machine learning' in text or 'ml ' in text: return 'Machine Learning Engineer'
    elif 'ai ' in text or 'artificial intelligence' in text: return 'AI Engineer'
    # Web & Software
    elif 'frontend' in text or 'front-end' in text: return 'Frontend Developer'
    elif 'backend' in text or 'back-end' in text: return 'Backend Developer'
    elif 'fullstack' in text or 'full stack' in text: return 'Full Stack Developer'
    elif 'web developer' in text or 'web dev' in text: return 'Web Developer'
    # Mobile
    elif 'android' in text: return 'Android Developer'
    elif 'ios' in text: return 'iOS Developer'
    elif 'mobile' in text or 'flutter' in text: return 'Mobile Developer (General)'
    # Infrastructure & Security
    elif 'devops' in text: return 'DevOps Engineer'
    elif 'cloud' in text or 'aws' in text: return 'Cloud Engineer'
    elif 'cyber' in text or 'security' in text or 'hacker' in text: return 'Cyber Security'
    elif 'network' in text: return 'Network Engineer'
    elif 'it support' in text or 'support engineer' in text: return 'IT Support'
    # Design, QA, & Specialized
    elif 'ui/ux' in text or 'ui ' in text or 'ux ' in text or 'designer' in text: return 'UI/UX Designer'
    elif 'product manager' in text: return 'Product Manager'
    elif 'qa' in text or 'quality assurance' in text or 'tester' in text: return 'QA Engineer'
    elif 'architect' in text: return 'Software Architect'
    elif 'game' in text or 'unity' in text: return 'Game Developer'
    elif 'blockchain' in text or 'web3' in text: return 'Blockchain Developer'
    elif 'embedded' in text or 'iot' in text or 'robotics' in text: return 'Embedded Systems Engineer'
    elif 'writer' in text or 'copywriter' in text: return 'Technical Writer'
    else: return 'Software Engineer (General)'

df_global['role_standard'] = df_global['job_title'].apply(map_27_roles)

# **Sel 9: Markdown**

### **5. Final Check & Save Dataset**

# **Sel 10: Code**

In [17]:
# Ambil kolom pertama dari vocab sebagai daftar skill standar
col_vocab = df_vocab.columns[0]
standard_skills = df_vocab[col_vocab].astype(str).str.lower().tolist()

def extract_skills(text, vocab):
    text = str(text).lower()
    found = [skill for skill in vocab if skill in text]
    return ", ".join(list(set(found))) # Menggunakan set agar tidak duplikat

# Ekstraksi Skill
df_global['skills_cleaned'] = df_global['raw_skills'].apply(lambda x: extract_skills(x, standard_skills))

# Feature Engineering: Hitung jumlah skill
df_global['skills_count'] = df_global['skills_cleaned'].apply(lambda x: len(x.split(', ')) if x != "" else 0)

# **Sel 6: Final Check & Export CSV**

In [18]:
# Hapus kolom pembantu
df_final = df_global.drop(columns=['raw_skills'])

# Tampilkan preview untuk memastikan semua role (Mobile, QA, dll) muncul
print("Sample Distribusi Role:")
print(df_final['role_standard'].value_counts().head(10))

# Simpan ke CSV untuk dipakai di EDA.ipynb
df_final.to_csv('global_final_for_eda.csv', index=False)

print("\nBERES! File 'global_final_for_eda.csv' sudah siap dipakai di EDA.ipynb 🚀")
display(df_final.head())

Sample Distribusi Role:
role_standard
Software Engineer (General)    222
Web Developer                   83
Cyber Security                  70
Data Scientist                  69
DevOps Engineer                 57
UI/UX Designer                  37
QA Engineer                     20
AI Engineer                     19
Technical Writer                19
Cloud Engineer                  14
Name: count, dtype: int64

BERES! File 'global_final_for_eda.csv' sudah siap dipakai di EDA.ipynb 🚀


,job_title,role_standard,skills_cleaned,skills_count
0,Cybersecurity,Cyber Security,"docker, r, node, go, pandas, c, mongodb",7
1,Cybersecurity,Cyber Security,"css, javascript, r, sql, java, c",6
2,DevOps,DevOps Engineer,"css, javascript, r, go, java, c, mongodb, kube...",9
3,Web Dev,Web Developer,"javascript, r, sql, java, c",5
4,Cybersecurity,Cyber Security,"python, docker, r, node, go, pandas, c, mongod...",9
